# **Basket Analysis & Recommendation Engine (Association Rules)**

This notebook performs a comprehensive Market Basket Analysis (MBA) on sales data to uncover purchasing patterns and generate product recommendations. Key aspects covered include:

*   **Data Loading and Cleaning**: Initial setup, data cleaning, and parameter definition for the analysis.
*   **Product-Level MBA**: Analysis of co-purchase patterns at a fine-grained product sub-category level.
*   **Category-Level MBA**: Analysis of broader co-purchase patterns at the commercial category level.
*   **Temporal Analysis**: Investigation of sequential buying behaviors and customer journeys over a 30-day window.
*   **Operative Brand Analysis**: Segmentation of the MBA by specific brands ('Versace Accessories' and 'Versace Young') to identify brand-specific patterns.
*   **Geographical Analysis**: Comparison of association rules across different countries to identify universal vs. country-specific purchasing behaviors.

We have then evaluated the results obtained for the analysis

#**Session 1 — Data loading, cleaning, and parameter setup**

In this section, we set up the Market Basket Analysis workflow by importing the necessary libraries, loading the dataset, and defining the key thresholds used to extract association rules at both the product and category levels.

The choice of parameters reflects a trade-off between rule coverage, statistical reliability, and interpretability. In particular, the minimum support threshold at the product level and category level was calibrated to capture patterns that occur with sufficient frequency (approximately corresponding to at least 15 transactions). However, in cases where the data becomes more fragmented—such as country-level analyses with smaller sample sizes—this threshold was relaxed to ensure that meaningful patterns are not discarded simply due to limited observations. This adjustment allows us to retain relevant associations while acknowledging the sparsity of the data.

At the category level, thresholds were set more conservatively ( confidence = 0.10). This is because broader categories naturally occur more frequently and tend to generate denser association structures. Stricter thresholds help filter out trivial or overly generic relationships, ensuring that the resulting rules remain informative. In both analyses, the minimum lift was fixed at 1.00 to retain only positive associations that exceed random co-occurrence, while the output was restricted to the top five recommendations per focal item to maintain clarity and usability.

The analysis mainly distinguishes between two levels of granularity: commercial_sub_category is used as a proxy for individual products, while commercial_category captures broader product groupings. This distinction allows us to compare fine-grained behavioral patterns with more aggregated purchasing logic.

Before running the analysis, the dataset is preprocessed to retain only the relevant variables (transaction_id, commercial_sub_category, and commercial_category), remove missing values, and standardize text fields. Duplicate item occurrences within the same transaction are eliminated to avoid artificially inflating co-occurrence frequencies. Finally, only baskets containing at least two distinct items or categories are kept, since association rule mining requires multi-item transactions to identify meaningful relationships.

In [ ]:
import pandas as pd
import numpy as np
import warnings
from mlxtend.frequent_patterns import fpgrowth, association_rules

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# 0. PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
INPUT_FILE = "IMA2026_Versace_MBA_Product_Level.csv"

MIN_CONFIDENCE_PRODUCT = 0.05
MIN_LIFT_PRODUCT = 1.00

MIN_CONFIDENCE_CATEGORY = 0.10
MIN_LIFT_CATEGORY = 1.00

TOP_N_RECOMMENDATIONS = 5

# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD & CLEAN
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print("VERSACE — MARKET BASKET ANALYSIS")
print("=" * 80)

df_complete = pd.read_csv(INPUT_FILE)

# Columns to be cleaned in df_complete
cols_to_clean = [
    "transaction_id",
    "commercial_sub_category",
    "commercial_category"
]

# Apply cleaning steps directly to df_complete
df_complete = df_complete.dropna(subset=cols_to_clean)
df_complete["transaction_id"] = df_complete["transaction_id"].astype(str).str.strip()
df_complete["commercial_sub_category"] = df_complete["commercial_sub_category"].astype(str).str.strip()
df_complete["commercial_category"] = df_complete["commercial_category"].astype(str).str.strip()

# Create df from the cleaned df_complete, selecting only the needed columns
df = df_complete[cols_to_clean].copy()
print(df.head(5))

print(f"Rows: {len(df):,}")
print(f"Transactions: {df['transaction_id'].nunique():,}")
print(f"Unique products (sub-categories): {df['commercial_sub_category'].nunique():,}")
print(f"Unique categories: {df['commercial_category'].nunique():,}")


# Remove duplicates within transaction
df_product = df.drop_duplicates(subset=["transaction_id", "commercial_sub_category"]).copy()
df_category = df.drop_duplicates(subset=["transaction_id", "commercial_category"]).copy()

# Keep only multi-item transactions for product-level
prod_sizes = df_product.groupby("transaction_id")["commercial_sub_category"].nunique()
multi_prod_tids = prod_sizes[prod_sizes >= 2].index
df_product_multi = df_product[df_product["transaction_id"].isin(multi_prod_tids)].copy()

# Keep only multi-category transactions for category-level
cat_sizes = df_category.groupby("transaction_id")["commercial_category"].nunique()
multi_cat_tids = cat_sizes[cat_sizes >= 2].index
df_category_multi = df_category[df_category["transaction_id"].isin(multi_cat_tids)].copy()

total_unique_transactions = df['transaction_id'].nunique()

num_multi_prod_transactions = df_product_multi['transaction_id'].nunique()
percentage_multi_prod_transactions = (num_multi_prod_transactions / total_unique_transactions) * 100

num_multi_cat_transactions = df_category_multi['transaction_id'].nunique()
percentage_multi_cat_transactions = (num_multi_cat_transactions / total_unique_transactions) * 100


print(f"\nMulti-product transactions: {num_multi_prod_transactions:,}")
print(f"Percentage of multi-product transactions: {percentage_multi_prod_transactions:.2f}%")

print(f"Multi-category transactions: {num_multi_cat_transactions:,}")
print(f"Percentage of multi-category transactions: {percentage_multi_cat_transactions:.2f}%")

print(df_complete.head(5))
df.head(5)

#**Session 2 — Helper functions for basket construction, rule generation, and recommendation output**

In this section, we define the helper functions that structure the core logic of the analysis.

First, build_one_hot_basket() transforms the transactional dataset into a one-hot encoded basket matrix, where each row represents a transaction and each column represents a product or category. A value of True indicates that the item is present in the basket. This format is required for FP-Growth.

Next, generate_rules() performs the association rule mining. It first applies the FP-Growth algorithm to identify frequent itemsets above the chosen minimum support threshold. It then generates association rules using confidence as the filtering metric. To keep the output clear and directly interpretable, the function retains only simple one-to-one rules of the form A → B, excluding more complex combinations involving multiple antecedents or consequents. It then extracts the focal and associated items, filters the rules by minimum lift, and keeps only the final metrics required for the analysis: probability (confidence), support, and lift.

The build_output_table() function reformats the rules into a cleaner summary table by renaming the columns according to the level of analysis and sorting the results so that the strongest associations appear first for each focal item.

Finally, build_recommendation_table() creates a more compact recommendation output by selecting only the top-N associated items for each focal product or category, based on probability, support, and lift, and assigning a recommendation rank.

Overall, this section builds the reusable functions needed to transform cleaned transactions into association rules and final recommendation tables in a structured and interpretable way.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────
def build_one_hot_basket(data, transaction_col, item_col):
    basket = (
        data.assign(value=1)
        .pivot_table(
            index=transaction_col,
            columns=item_col,
            values="value",
            aggfunc="max",
            fill_value=0
        )
        .astype(bool)
    )
    return basket

def generate_rules(basket_matrix, min_support, min_confidence, min_lift):
    freq_itemsets = fpgrowth(
        basket_matrix,
        min_support=min_support,
        use_colnames=True
    )

    if freq_itemsets.empty:
        return pd.DataFrame()

    rules = association_rules(
        freq_itemsets,
        metric="confidence",
        min_threshold=min_confidence
    )

    if rules.empty:
        return pd.DataFrame()

    # Keep only simple pairwise rules: A -> B
    rules = rules[
        (rules["antecedents"].apply(len) == 1) &
        (rules["consequents"].apply(len) == 1)
    ].copy()

    if rules.empty:
        return pd.DataFrame()

    rules["focal_item"] = rules["antecedents"].apply(lambda x: list(x)[0])
    rules["associated_item"] = rules["consequents"].apply(lambda x: list(x)[0])

    rules = rules[rules["lift"] >= min_lift].copy()

    # Keep exactly the metrics you want
    rules = rules.rename(columns={
        "confidence": "probability",
        "support": "support"
    })

    rules = rules[[
        "focal_item",
        "associated_item",
        "probability",
        "support",
        "lift"
    ]].copy()

    return rules

def build_output_table(rules_df, focal_name, assoc_name):
    if rules_df.empty:
        return rules_df

    out = rules_df.rename(columns={
        "focal_item": focal_name,
        "associated_item": assoc_name
    }).copy()

    # nicer formatting order
    out = out.sort_values(
        by=[focal_name, "probability", "support", "lift"],
        ascending=[True, False, False, False]
    ).reset_index(drop=True)

    return out

def build_recommendation_table(summary_df, focal_col, assoc_col, top_n=5):
    if summary_df.empty:
        return summary_df

    rec = (
        summary_df.sort_values(
            by=[focal_col, "probability", "support", "lift"],
            ascending=[True, False, False, False]
        )
        .groupby(focal_col, as_index=False)
        .head(top_n)
        .copy()
    )

    rec["rank"] = rec.groupby(focal_col).cumcount() + 1
    rec = rec[[focal_col, "rank", assoc_col, "probability", "support", "lift"]]
    return rec


#**Session 3 — Product-level association analysis**

In this section, we perform Market Basket Analysis at the product level using **commercial_sub_category** as the unit of analysis. This approach is consistent with the broader analytical framework adopted throughout Sessions 3 to 8, where the focus is on transforming transactional data into structured representations that allow us to uncover meaningful behavioral patterns.

The cleaned transactional data is first converted into a one-hot encoded basket matrix, where each row represents a transaction and each column corresponds to a product sub-category. This transformation enables the identification of co-occurrence patterns across items within the same basket.

Association rules are then generated using predefined thresholds for support, confidence, and lift. These thresholds are selected to balance statistical reliability with interpretability, ensuring that the extracted rules are both meaningful and actionable. The resulting rules are reorganized into a structured summary table reporting, for each focal product, the associated products along with the key metrics: probability (confidence), support, and lift. A recommendation table is subsequently built by retaining only the top associations for each focal product.

In line with the general analytical principles used across Sessions 3 to 8, interpretation prioritizes **support first**, followed by **probability**, and finally **lift**. Support is emphasized because it reflects the overall frequency and therefore the business relevance of a relationship. Probability captures the predictive strength of the rule, indicating how likely the associated item is given the focal product. Lift serves as a validation metric, ensuring that the observed association is stronger than what would be expected by chance.

Overall, the most valuable rules are those combining **high support, strong probability, and lift greater than one**, as they represent associations that are both statistically meaningful and commercially impactful.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. PRODUCT-LEVEL = commercial_sub_category
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 80)
print("SECTION 1 — PRODUCT LEVEL (commercial_sub_category)")
print("─" * 80)

basket_product = build_one_hot_basket(
    data=df_product_multi,
    transaction_col="transaction_id",
    item_col="commercial_sub_category"
)

MIN_SUPPORT_PRODUCT = 15/num_multi_prod_transactions


rules_product = generate_rules(
    basket_matrix=basket_product,
    min_support=MIN_SUPPORT_PRODUCT,
    min_confidence=MIN_CONFIDENCE_PRODUCT,
    min_lift=MIN_LIFT_PRODUCT
)

summary_product = build_output_table(
    rules_df=rules_product,
    focal_name="focal_product",
    assoc_name="associated_product"
)

recommendations_product = build_recommendation_table(
    summary_df=summary_product,
    focal_col="focal_product",
    assoc_col="associated_product",
    top_n=TOP_N_RECOMMENDATIONS
)

print(f"Product-level rules: {len(summary_product):,}")


recommendations_product=recommendations_product.sort_values(by=['support', 'probability', 'lift'], ascending=False)
recommendations_product

This analysis identifies co-purchase patterns at the **product sub-category level**, revealing how customers combine items within their baskets.

**Most Relevant Associations**

The results highlight a few **strong and intuitive relationships**, notable for their **consistent support** alongside high lift and probability:

*   **Homewear Complementarity**
    - *Slippers ↔ Bathrobes* : Probability ≈ 0.60–0.70 , Support ≈ 5.1% , Lift ≈ 8.17
    - → Strong and **bidirectional relationship**, indicating a clear “comfort/home set” bundle, validated by its significant support.

*   **Casual Outfit Pairing**
    - *Short Pants ↔ Informal Shirt* : Probability ≈ 0.60 , Support ≈ 2.9% , Lift ≈ 7.06
    - → Clear **outfit-based purchasing behavior**, with reasonable support making it commercially relevant.

*   **Weak Apparel Cross-Selling**
    - *Sneakers ↔ T-Shirt* : Probability ≈ 0.20–0.26, Support ≈ 3.6% , Lift ≈ 1.43
    - → Present but **relatively weak association** (low lift), despite moderate support. This suggests more independent purchases rather than strong bundling.

**Lower-Relevance Associations**

*   Categories like *Sun*, *Other*, *Shoulder*, *Sandals*:
    - Low probability (~ 0.05 – 0.07), Lift close to 1, and often minimal support (~0.4%)
    - → These relationships are **weak or not meaningful**, particularly due to their low support.

**Managerial Implications**

*   Focus recommendations on:
    - **High-lift, intuitive bundles with solid support** (homewear, outfits) as these are the most actionable and reliable due to their frequency and strength.
*   Deprioritize:
    - Low-lift associations (≈1), as they do not indicate real behavioral patterns.
    - Rules with **very low support**, even if lift is high, as they may represent niche behaviors with limited commercial impact.

#**Session 4 — Category-level association analysis**



This section delves into Market Basket Analysis at the category level, using `commercial_category` as the unit of analysis. We explore broader co-purchase patterns among product groups, aiming to identify strategic insights for cross-category recommendations and bundling.

In [ ]:
MIN_SUPPORT_CATEGORY = 15/num_multi_cat_transactions
MIN_SUPPORT_CATEGORY

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. CATEGORY-LEVEL = commercial_category
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 80)
print("SECTION 2 — CATEGORY LEVEL (commercial_category)")
print("─" * 80)

basket_category = build_one_hot_basket(
    data=df_category_multi,
    transaction_col="transaction_id",
    item_col="commercial_category"
)


rules_category = generate_rules(
    basket_matrix=basket_category,
    min_support=MIN_SUPPORT_CATEGORY,
    min_confidence=MIN_CONFIDENCE_CATEGORY,
    min_lift=MIN_LIFT_CATEGORY
)

summary_category = build_output_table(
    rules_df=rules_category,
    focal_name="focal_category",
    assoc_name="associated_category"
)

recommendations_category = build_recommendation_table(
    summary_df=summary_category,
    focal_col="focal_category",
    assoc_col="associated_category",
    top_n=TOP_N_RECOMMENDATIONS
)

print(f"Category-level rules: {len(summary_category):,}")

recommendations_category_1=recommendations_category.sort_values(by=['support', 'probability', 'lift'], ascending=False)
recommendations_category_1.head(10)

This analysis examines co-purchase patterns at the **category level**, providing a broader view of how customers combine different types of products.

**Most Relevant Associations**

The results reveal **clear and interpretable cross-category relationships**, where **support indicates their prevalence**:

*   **Outfit Construction (Strongest Pattern)**
    - *Pants ↔ Shirt*: Probability ≈ 0.33–0.47, Support ≈ 5.5%, **Lift ≈ 2.81**
    - → Strong evidence of **top–bottom pairing**, a core purchasing behavior, and frequent enough to be commercially significant.

*   **Apparel Extensions**
    - *Pants ↔ Jersey*: Probability 0.33- 0.15, Support ≈ 4.0%, Lift ≈ 1.23
    - → Suggests **layering or casual outfit combinations**, but weaker than Shirt–Pants. Its moderate support still indicates a noticeable pattern.

*   **Accessories / Complementary Items**
    - *Slg (Small Leather Goods) ↔ Bags*: Probability 0.4-0.22, Support ≈ 3.5%, Lift ≈ 2.55
    - → Indicates **accessory bundling behavior**, a moderately frequent and strong association.

*   **Seasonal / Usage-Based Pairing**
    - *Underwear ↔ Beachwear*: Probability: 0.29 - 0.19, **Support ≈ 2.8%**, Lift ≈ 1.97
    - → Reflects **context-driven purchases** (e.g., travel, summer), showing a consistent, albeit less frequent, pattern.

**Weaker or Ambiguous Associations**

*   *Bags ↔ Shoes*: Lift ≈ 1.03
    - Despite relatively high probability (~0.39) and highest support (≈6.2%) among category rules, the low lift suggests **no real association beyond baseline frequency**. While frequent, these items are purchased together simply because they are popular, not necessarily as a strong pairing.

**Managerial Implications**

*   Prioritize recommendations based on:
    - **Outfit logic with strong support** (core driver of behavior)
    - **Accessory bundling with good support**
    - These are frequent, strong, and actionable patterns.
*   Be cautious with:
    - High-frequency categories where lift is low (e.g., Bags ↔ Shoes) → **not true signals** for cross-selling, despite high support. The high support merely indicates popularity, not a strong associative bond.

# **Session 5 - Temporal Analysis**

This session extends our Market Basket Analysis to incorporate a temporal dimension. By analyzing co-purchases within a 30-day window, we aim to uncover sequential buying behaviors and customer journeys that occur over multiple interactions, rather than just single transactions. This provides insights into how customers build their baskets over time.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SESSION 5.1 — TEMPORAL USER-BASED BASKETS (30-Day Window)
# ─────────────────────────────────────────────────────────────────────────────

# 1. Convert date and sort
df_complete['event_date'] = pd.to_datetime(df_complete['event_date'])
df_complete = df_complete.sort_values(['user_pseudo_id', 'event_date'])

# 2. Create a 'User-Month' identifier (Simplest 30-day proxy)
# This groups all transactions of a user within the same calendar month
df_complete['user_period_id'] = df_complete['user_pseudo_id'].astype(str) + "_" + df_complete['event_date'].dt.to_period('M').astype(str)

print(f"Original Transactions: {df_complete['transaction_id'].nunique():,}")
print(f"Aggregated User-Period Baskets: {df_complete['user_period_id'].nunique():,}")

# ─────────────────────────────────────────────────────────────────────────────
# TEMPORAL PRODUCT-LEVEL = commercial_sub_category
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 80)
print("SECTION 5.1 — TEMPORAL PRODUCT LEVEL (commercial_sub_category)")
print("─" * 80)

# 1. Build the new matrix using 'user_period_id'
basket_temporal_product = build_one_hot_basket(
    data=df_complete.drop_duplicates(subset=['user_period_id','commercial_sub_category']),
    transaction_col="user_period_id",
    item_col="commercial_sub_category"
)

# 2. Filter for multi-item baskets (now includes items bought in different sessions!)
temporal_multi_product = basket_temporal_product[basket_temporal_product.sum(axis=1) >= 2]
print(f"Multi-item Temporal Product Baskets: {len(temporal_multi_product):,}")

MIN_SUPPORT_TEMPORAL_PRODUCT = 15/len(temporal_multi_product)
print(f"MIN_SUPPORT_TEMPORAL_PRODUCT: {MIN_SUPPORT_TEMPORAL_PRODUCT}")

# 3. Run Rules (using the new optimized support you found in the Elbow Analysis)
rules_temporal_product = generate_rules(
    basket_matrix=temporal_multi_product,
    min_support=MIN_SUPPORT_TEMPORAL_PRODUCT,
    min_confidence=0.05,
    min_lift=1.0
)

# 4. Prepare summary table to rename columns as expected by build_recommendation_table
summary_temporal_product = build_output_table(
    rules_df=rules_temporal_product,
    focal_name="focal_product",
    assoc_name="associated_product"
)

print(f"Temporal Product-level rules: {len(summary_temporal_product):,}")

recommendations_temporal_product = build_recommendation_table(
    summary_df=summary_temporal_product,
    focal_col="focal_product",
    assoc_col="associated_product"
)
print("\nTop 10 Temporal Product Recommendations:")
display(recommendations_temporal_product.head(10))

# ─────────────────────────────────────────────────────────────────────────────
# TEMPORAL CATEGORY-LEVEL = commercial_category
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 80)
print("SECTION 5.2 — TEMPORAL CATEGORY LEVEL (commercial_category)")
print("─" * 80)

# 1. Build the new matrix using 'user_period_id'
basket_temporal_category = build_one_hot_basket(
    data=df_complete.drop_duplicates(subset=['user_period_id','commercial_category']),
    transaction_col="user_period_id",
    item_col="commercial_category"
)

# 2. Filter for multi-category baskets
temporal_multi_category = basket_temporal_category[basket_temporal_category.sum(axis=1) >= 2]
print(f"Multi-item Temporal Category Baskets: {len(temporal_multi_category):,}")

MIN_SUPPORT_TEMPORAL_CATEGORY = 15/len(temporal_multi_category)
print(f"MIN_SUPPORT_TEMPORAL_CATEGORY: {MIN_SUPPORT_TEMPORAL_CATEGORY}")

# 3. Run Rules
rules_temporal_category = generate_rules(
    basket_matrix=temporal_multi_category,
    min_support=MIN_SUPPORT_TEMPORAL_CATEGORY,
    min_confidence=0.10,
    min_lift=1.0
)

# 4. Prepare summary table
summary_temporal_category = build_output_table(
    rules_df=rules_temporal_category,
    focal_name="focal_category",
    assoc_name="associated_category"
)

print(f"Temporal Category-level rules: {len(summary_temporal_category):,}")

recommendations_temporal_category = build_recommendation_table(
    summary_df=summary_temporal_category,
    focal_col="focal_category",
    assoc_col="associated_category"
)
print("\nTop 10 Temporal Category Recommendations:")
display(recommendations_temporal_category.head(10))

This analysis captures **co-purchases across time (within 30 days)**, revealing how customers progressively build their baskets. In this temporal context, **support indicates the overall frequency of these multi-session buying sequences.**

**Product-Level Patterns**

*   **Strong and persistent homewear bundle**
    - *Bathrobes → Slippers* (Support = 4.97%, Probability = 0.567, Lift = 7.54)
    - → Confirms a **robust complementarity both within and across transactions**, consistently showing high support and lift.

*   **Intimate apparel complementarity**
    - *Bra → Panties* (Support = 1.80%, Probability = 0.589, Lift = 7.90)
    - → Stable and strong pairing, independent of timing, with consistent support for such a fundamental pairing.

*   **Sequential outfit building**
    - *Blouson → T-Shirt* (Support = 0.44%, Probability = 0.238, Lift = 1.28)
    - *Boots/Booties → Sneakers* (Support = 0.44%, Probability = 0.196, Lift = 1.31)
    - → Customers **complete outfits over multiple sessions**, not only in one basket. Support for these sequences highlights their commercial relevance over time.

*   **Notable anomaly**
    - *Bedroom → Living Room* (Support = 0.35%, Lift = 58.30)
    - → Likely driven by **low frequency or niche behavior**, not a robust pattern. The extremely high lift here is misleading due to its rare occurrence (low support).

**Category-Level Patterns**

*   **Consistent outfit logic over time**
    - *Denim/Jeans → Jersey / Shirt / Pants*
    - *Dress → Jersey / Shirt*
    - → Clear **step-by-step outfit construction**. The support for these category sequences emphasizes their recurring nature.

*   **Sportswear cluster**
    - *Activewear → Jersey* (Support = 0.54%, Probability = 0.800, Lift = 2.88)
    - → Strong and focused behavior within category, occurring with moderate support in temporal baskets.

*   **Accessories continuity**
    - *Bags → SLG* (Support = 3.91%, Probability = 0.238, Lift = 2.42)
    - → Accessories are often completed in subsequent purchases, with a solid support indicating this is a common journey.

*   **Seasonal behavior**
    - *Beachwear → Underwear* (Support = 3.07%, Probability = 0.198, Lift = 1.90)
    - → Suggests planned or occasion-driven purchases, showing a meaningful level of occurrence.

**Comparison with Same-Transaction Analysis**

*   **Stable core patterns**
    - Homewear (Bathrobes–Slippers)
    - Outfit logic (Denim–Top, Pants–Shirt)
    - Accessories (Bags–SLG)
    → These are **structural behaviors**, independent of timing, and tend to maintain **meaningful support** across both types of analysis.

*   **Key difference**
    - Temporal analysis reveals **sequencing**:
        - Items are often **not bought together**, but within a short time window
        - Especially true for **apparel and accessories**. Rules with **moderate to high support in temporal analysis** are particularly valuable here, as they uncover genuine customer journeys that single-transaction analysis would miss.

*   Very high lift values with low support should be treated cautiously, as they represent rare events rather than robust, frequent patterns.

**Key Takeaways**

*   Customers frequently **spread purchases over time**, especially for:
    - Outfit completion
    - Accessories
    - These sequences are often observed with measurable support, indicating their commercial potential.

*   Same-transaction rules capture **instant bundles**, while temporal rules capture:
    - **purchase journeys**
    - **follow-up needs**
    - The support metric is crucial for determining the actionable value of these temporal patterns.

*   Very high lift values with low support should be treated cautiously, as they represent rare events rather than robust, frequent patterns.

## **Session 6 - Operative Brand Analysis**


In this session, we segment the Market Basket Analysis by specific operative brands. This approach allows us to identify brand-specific co-purchase patterns and to understand whether recommendation strategies need to be tailored for different brand portfolios.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SESSION 6 — SEGMENTED MBA: ACCESSORIES VS. YOUNG
# ─────────────────────────────────────────────────────────────────────────────

# 1. Define the brands to analyze
brands = ["Versace Accessories", "Versace Young"]

# Dictionary to store results
segmented_results = {}

print("Starting Segmented Analysis...")

for brand in brands:
    print(f"\nProcessing Brand: {brand}")

    # 2. Filter the dataframe for the specific brand
    # We use the original 'df' but ensure we only take the rows for this brand
    df_brand = df_complete[df_complete['operative_brand'] == brand].copy()

    # 3. Apply the 'Multi-Item' logic specific to this brand's transactions
    # This is crucial because a transaction might have 3 items,
    # but maybe only 1 belongs to the brand being analyzed.
    brand_prod_counts = df_brand.groupby("transaction_id")["commercial_sub_category"].nunique()
    multi_tids = brand_prod_counts[brand_prod_counts >= 2].index
    df_brand_multi = df_brand[df_brand["transaction_id"].isin(multi_tids)].copy()

    print(f" - Transactions for {brand}: {df_brand['transaction_id'].nunique():,}")
    print(f" - Multi-item {brand} baskets: {len(multi_tids):,}")

    if len(multi_tids) > 0:
        # Compute MIN_SUPPORT for this brand based on multi-item baskets
        MIN_SUPPORT_BRAND = 15 / len(multi_tids)
        print(f" - Calculated MIN_SUPPORT for {brand}: {MIN_SUPPORT_BRAND:.6f}")

        # 4. Build the Basket Matrix for this brand
        basket_brand = build_one_hot_basket(
            data=df_brand_multi,
            transaction_col="transaction_id",
            item_col="commercial_sub_category"
        )

        # 5. Generate Rules
        # Note: We use the optimized support dynamically calculated for each brand
        rules_brand = generate_rules(
            basket_matrix=basket_brand,
            min_support=MIN_SUPPORT_BRAND,
            min_confidence=MIN_CONFIDENCE_PRODUCT,
            min_lift=MIN_LIFT_PRODUCT
        )

        # 6. Format Output
        summary_brand = build_output_table(
            rules_df=rules_brand,
            focal_name="focal_product",
            assoc_name="associated_product"
        )

        recs_brand = build_recommendation_table(
            summary_df=summary_brand,
            focal_col="focal_product",
            assoc_col="associated_product",
            top_n=3
        )

        segmented_results[brand] = recs_brand
        print(f" - Rules generated for {brand}: {len(summary_brand)}")
    else:
        print(f" - Not enough data to generate rules for {brand}.")

# 7. Display Top Results for comparison
if "Versace Accessories" in segmented_results:
    print("\n--- TOP RULES: VERSACE ACCESSORIES ---")
    display(segmented_results["Versace Accessories"].head(10))

if "Versace Young" in segmented_results:
    print("\n--- TOP RULES: VERSACE YOUNG ---")
    display(segmented_results["Versace Young"].head(10))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SESSION 6  — BASKET SEGMENTATION | CATEGORY LEVEL
# ─────────────────────────────────────────────────────────────────────────────

# 1. Define the brands to analyze
brands_to_analyze = ["Versace Accessories", "Versace Young"]

# Dictionary to store results
segmented_results_category = {}

print("Starting Segmented Category-Level Analysis...")

for brand in brands_to_analyze:
    print(f"\nProcessing Brand: {brand}")

    # 2. Filter the dataframe for the specific brand
    # Use df_complete which has 'operative_brand' and 'commercial_category'
    df_brand_filtered = df_complete[df_complete['operative_brand'] == brand].copy()

    # 3. Apply the 'Multi-Category' logic specific to this brand's transactions
    # A transaction might have items from different categories, but we are interested
    # in multi-category transactions within the context of THIS brand.
    brand_cat_counts = df_brand_filtered.groupby("transaction_id")["commercial_category"].nunique()
    multi_cat_tids_brand = brand_cat_counts[brand_cat_counts >= 2].index
    df_brand_multi_cat = df_brand_filtered[df_brand_filtered["transaction_id"].isin(multi_cat_tids_brand)].copy()

    print(f" - Transactions for {brand}: {df_brand_filtered['transaction_id'].nunique():,}")
    print(f" - Multi-category {brand} baskets: {len(multi_cat_tids_brand):,}")

    if len(multi_cat_tids_brand) > 0:
        MIN_SUPPORT_BRAND_CATEGORY = 15 / len(multi_cat_tids_brand)
        print(f" - Calculated MIN_SUPPORT for {brand} (Category Level): {MIN_SUPPORT_BRAND_CATEGORY:.6f}")


        # 4. Build the Basket Matrix for this brand at the category level
        basket_brand_category = build_one_hot_basket(
            data=df_brand_multi_cat,
            transaction_col="transaction_id",
            item_col="commercial_category"
        )

        # 5. Generate Rules
        rules_brand_category = generate_rules(
            basket_matrix=basket_brand_category,
            min_support=MIN_SUPPORT_BRAND_CATEGORY,
            min_confidence=MIN_CONFIDENCE_CATEGORY,
            min_lift=MIN_LIFT_CATEGORY
        )

        # 6. Format Output
        summary_brand_category = build_output_table(
            rules_df=rules_brand_category,
            focal_name="focal_category",
            assoc_name="associated_category"
        )

        recs_brand_category = build_recommendation_table(
            summary_df=summary_brand_category,
            focal_col="focal_category",
            assoc_col="associated_category",
            top_n=5 # Using 5 recommendations as before
        )

        segmented_results_category[brand] = recs_brand_category
        print(f" - Rules generated for {brand}: {len(summary_brand_category)}")
    else:
        print(f" - Not enough data to generate rules for {brand}.")

# 7. Display Top Results for comparison
for brand in brands_to_analyze:
    if brand in segmented_results_category:
        print(f"\n--- TOP CATEGORY RULES: {brand.upper()} ---")
        display(segmented_results_category[brand])


This analysis segments the Market Basket Analysis by brand ('Versace Accessories' and 'Versace Young') to uncover brand-specific co-purchase patterns at both product and category levels. **The support metric is crucial here for assessing the commercial significance of each brand-specific rule.**

**Product-Level Analysis**

*   **Versace Accessories**:
    *   **Rules Found**: 30 product-level association rules were generated, with several strong and interpretable top associations.
    *   **Top Associations** (ordered by support, then probability, then lift):
        - **Belt ↔ Drivers** (Probability: 0.16–0.37, Support ≈ 2.2%, Lift: 2.64)
          → A strong *bidirectional* relationship, indicating complementary purchasing (likely styling bundles), occurring with sufficient support to be relevant.
        - **Bracelet → Necklace / Earrings** (Lift up to 2.72) - (Requires specific support values for clearer interpretation, but implies consistent pairing within the brand.)
        → Clear jewelry bundling behavior, indicating frequent co-purchase within this segment.
        - **Earrings → Necklace / Ring / Bracelet**
        → Reinforces cross-selling opportunities within accessories, with implied consistent support for these types of combinations.
        - **Boots → Sneakers** and **Clutch → Pouch**
        → Suggest substitution or multi-item basket behavior within similar use-cases.

    *   **Insight**:
        Accessories show **strong intra-category complementarities**, especially within jewelry and styling items. These are often backed by **good support values** within the Versace Accessories segment, suggesting effective opportunities for **bundling, recommendations, and cross-selling at the product level**.

*   **Versace Young**:
    *   **Rules Found**: 10 product-level association rules were generated.
    *   **Top Associations** (ordered by support, then probability, then lift):
        1.  **Informal Shirt ↔ Short Pants** (Support ≈ 14.6%, Probability: 0.60, Lift: 3.25)
            → This is the most frequent and strong outfit-based purchasing behavior for this brand.
        2.  **Sweatpants ↔ Sweatshirt / T-Shirt** (Support up to 9.1%, Lift up to 3.54)
            → Strong within-category casualwear bundles, showing a high frequency of co-occurrence.
        3.  **Gift Set ↔ Outdoor Blanket** (Support ≈ 5.2%, Probability: 0.44–0.63, Lift: 5.28)
            → Very strong pairing, likely seasonal or gifting-driven, with a meaningful level of support.

    *   **Insight**:
        'Versace Young' exhibits **coherent outfit-based purchasing patterns**, with very high lift values and solid support for key pairings, indicating strong and actionable cross-selling opportunities.

**Category-Level Analysis (commercial_category)**

*   **Versace Accessories**:
    *   **Rules Found**: 0 category-level association rules were generated.
    *   **Interpretation**:
        with Versace Accessories being primarily an accessories brand, many transactions likely contain only one category by definition, making 0 rules expected.

*   **Versace Young**:
    *   **Rules Found**: 10 category-level association rules were generated.
    *   **Top Associations** (ordered by support, then probability, then lift):
        1.  **Pants ↔ Shirt** (Support ≈ 20.9%, Lift: 2.04)
            → The most frequent cross-category outfit pairing, a strong indicator of core purchasing logic.
        2.  **Activewear → Jersey** (Support ≈ 7.1%, Probability: 0.90, Lift: 2.05)
            → Strong and frequent sportswear clustering within this brand.
        3.  **Dress ↔ Shoes** (Support ≈ 5.9%, Lift: 1.38)
            → Consistent apparel-footwear pairing, occurring with good frequency.
        4.  **Denim/Jeans → Jersey** (Support ≈ 7.1%, Lift: 1.86)
            → Another strong cross-category pairing for outfits.

    *   **Insight**:
        Clear **cross-category outfit logic** emerges, with **substantial support values** for these rules:
        - Tops ↔ Bottoms (Shirt–Pants)
        - Apparel ↔ Footwear (Dress–Shoes)
        - Sportswear clustering (Activewear–Jersey)

Overall, the brand-level segmentation reveals **two very different purchasing logics**:

*   **Versace Accessories**:
    - Strong **product-level complementarities** (especially within jewelry and styling items), with actionable support values.
    - Weak or absent **category-level structure**, indicating that cross-category purchasing for this brand does not reach sufficient support for clear rules.
    - → Best strategy: **fine-grained product recommendations and bundles** where support allows.

*   **Versace Young**:
    - Strong patterns at **both product and category levels**, characterized by **robust support for outfit-based purchasing behavior**.
    - Clear **outfit-based purchasing behavior**
    - → Best strategy: **cross-category styling recommendations (complete looks)**, leveraging the high support and lift of these rules.

Overall, this confirms that **consumer behavior is highly brand-dependent**, and effective recommendation systems or targeting strategies should be **customized at the brand level rather than applied uniformly**. The **support metric provides the necessary context for commercial viability** of these brand-specific insights.

## **Session 7 - Geographical Analysis**

In this session, we perform a Market Basket Analysis segmented by **country**. By analyzing association rules across different geographical regions, we aim to identify whether purchasing behaviors are universal or if there are significant country-specific patterns. This cross-country comparison can inform global vs. localized recommendation strategies, inventory management, and marketing campaigns. We will focus on the top countries by transaction volume to ensure meaningful and statistically robust results.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SESSION 7 — COUNTRY-LEVEL SEGMENTED MBA
# ─────────────────────────────────────────────────────────────────────────────

# 1. Identify the top countries by transaction volume to ensure meaningful results
top_countries = df_complete['country'].value_counts().head(5).index.tolist()
print(f"Analyzing Top Countries: {top_countries}")

# Dictionaries to store the results
country_results_subcat = {}
country_results_cat = {}

for country in top_countries:
    print(f"\n" + "="*40)
    print(f"PROCESSING COUNTRY: {country}")
    print("="*40)

    # Filter data for the specific country
    df_c = df_complete[df_complete['country'] == country].copy()

    # --- SUB-CATEGORY LEVEL (Product Level) ---
    # Identify multi-item transactions for this country
    subcat_counts = df_c.groupby("transaction_id")["commercial_sub_category"].nunique()
    multi_sub_tids = subcat_counts[subcat_counts >= 2].index
    df_c_sub = df_c[df_c["transaction_id"].isin(multi_sub_tids)].copy()

    if len(multi_sub_tids) > 10: # Minimum transaction threshold for the country
        basket_sub = build_one_hot_basket(df_c_sub, "transaction_id", "commercial_sub_category")

        # Dynamically calculate MIN_SUPPORT_SUBCAT for the current country - Attempt 1
        min_support_subcat_country_attempt1 = 15 / len(multi_sub_tids)
        print(f" - Calculated MIN_SUPPORT for {country} (Sub-category Level - Attempt 1): {min_support_subcat_country_attempt1:.6f}")

        rules_sub_attempt1 = generate_rules(
            basket_matrix=basket_sub,
            min_support=min_support_subcat_country_attempt1,
            min_confidence=MIN_CONFIDENCE_PRODUCT,
            min_lift=MIN_LIFT_PRODUCT
        )

        if len(rules_sub_attempt1) >= 10:
            rules_sub = rules_sub_attempt1
            min_support_subcat_country = min_support_subcat_country_attempt1
            print(f" - Kept MIN_SUPPORT_SUBCAT for {country}: {min_support_subcat_country:.6f} (Found {len(rules_sub)} rules)")
        else:
            # If less than 10 rules, recalculate with a lower support
            min_support_subcat_country = 5 / len(multi_sub_tids)
            print(f" - Recalculated MIN_SUPPORT for {country} (Sub-category Level - Attempt 2): {min_support_subcat_country:.6f}")
            rules_sub = generate_rules(
                basket_matrix=basket_sub,
                min_support=min_support_subcat_country,
                min_confidence=MIN_CONFIDENCE_PRODUCT,
                min_lift=MIN_LIFT_PRODUCT
            )
            print(f" - Found {len(rules_sub)} rules with adjusted MIN_SUPPORT.")

        if not rules_sub.empty:
            summary_sub = build_output_table(rules_sub, "focal_product", "associated_product")
            country_results_subcat[country] = build_recommendation_table(summary_sub, "focal_product", "associated_product")
            print(f" - {country}: Found {len(summary_sub)} sub-category rules.")
        else:
            print(f" - {country}: No sub-category rules found even with adjusted support.")

    # --- CATEGORY LEVEL ---
    # Identify multi-category transactions for this country
    cat_counts = df_c.groupby("transaction_id")["commercial_category"].nunique()
    multi_cat_tids = cat_counts[cat_counts >= 2].index
    df_c_cat = df_c[df_c["transaction_id"].isin(multi_cat_tids)].copy()

    if len(multi_cat_tids) > 10:
        basket_cat = build_one_hot_basket(df_c_cat, "transaction_id", "commercial_category")

        # Dynamically calculate MIN_SUPPORT_CAT for the current country - Attempt 1
        min_support_cat_country_attempt1 = 15 / len(multi_cat_tids)
        print(f" - Calculated MIN_SUPPORT for {country} (Category Level - Attempt 1): {min_support_cat_country_attempt1:.6f}")

        rules_cat_attempt1 = generate_rules(
            basket_matrix=basket_cat,
            min_support=min_support_cat_country_attempt1,
            min_confidence=MIN_CONFIDENCE_CATEGORY,
            min_lift=MIN_LIFT_CATEGORY
        )

        if len(rules_cat_attempt1) >= 10:
            rules_cat = rules_cat_attempt1
            min_support_cat_country = min_support_cat_country_attempt1
            print(f" - Kept MIN_SUPPORT_CAT for {country}: {min_support_cat_country:.6f} (Found {len(rules_cat)} rules)")
        else:
            # If less than 10 rules, recalculate with a lower support
            min_support_cat_country = 5 / len(multi_cat_tids)
            print(f" - Recalculated MIN_SUPPORT for {country} (Category Level - Attempt 2): {min_support_cat_country:.6f}")
            rules_cat = generate_rules(
                basket_matrix=basket_cat,
                min_support=min_support_cat_country,
                min_confidence=MIN_CONFIDENCE_CATEGORY,
                min_lift=MIN_LIFT_CATEGORY
            )
            print(f" - Found {len(rules_cat)} rules with adjusted MIN_SUPPORT.")

        if not rules_cat.empty:
            summary_cat = build_output_table(rules_cat, "focal_category", "associated_category")
            country_results_cat[country] = build_recommendation_table(summary_cat, "focal_category", "associated_category")
            print(f" - {country}: Found {len(summary_cat)} category rules.")
        else:
            print(f" - {country}: No category rules found even with adjusted support.")

# 2. Displaying results for all countries more explicitly to avoid truncation
for country in top_countries:
    if country in country_results_subcat and not country_results_subcat[country].empty:
        print(f"\n--- TOP SUB-CATEGORY RULES FOR {country.upper()} ---")
        print(country_results_subcat[country].head(10).to_markdown(index=False))

    if country in country_results_cat and not country_results_cat[country].empty:
        print(f"\n--- TOP CATEGORY RULES FOR {country.upper()} ---")
        print(country_results_cat[country].head(10).to_markdown(index=False))

This analysis compares association rules across countries (US, UK, Canada, Australia, Germany) to identify the most robust co-purchase patterns and highlight cross-country similarities and differences. **The varying levels of support across countries for similar rules provide insights into market size and the prevalence of purchasing behaviors in different regions.**

**Most Relevant Product-Level Patterns**

Several **strong and recurring associations** emerge across markets, with their **support reflecting their prevalence in each country**:

*   **Homewear / Comfort Items (Highly Consistent)**
    - *Bathrobes ↔ Slippers* appears in **US, UK, Australia, Germany**
    - Very high lift values (often >6, up to ~10)
    - Support varies by country (e.g., 5.2% in US, 3.5% in Germany)
    - → Represents a **universal “comfort bundle” behavior**, with its commercial importance directly tied to its support in each market.

*   **Casual Apparel Bundles (Consistent)**
    - *T-Shirt ↔ Polo / Sweatshirt / Sweatpants* (UK, Canada, Australia)
    - *Informal Shirt ↔ Short Pants* (Germany)
    - → Reflects **consistent outfit-based purchasing in casualwear**, with varying levels of support indicating different market sizes or preferences.

*   **Footwear Linkages (Moderately Consistent)**
    - *Shoes ↔ Apparel* relationships appear across multiple countries
    - → Footwear is often part of **multi-item outfit construction**. The **support for these rules can highlight the frequency of such bundling in specific regions.**

*   **Accessories (Localized)**
    - Strong jewelry bundling (e.g., *Bracelet → Earrings*) mainly in **US**
    - → Less visible or absent in other countries, indicating either lower frequency (support below threshold) or different market dynamics.

**Most Relevant Category-Level Patterns**

At category level, **clear and consistent structure emerges**, where **support underlines the frequency of cross-category purchases**:

*   **Top–Bottom Apparel Combinations (Highly Consistent)**
    - *Shirt ↔ Pants* (Canada, Germany with support ranging from ~4.5% to ~11.5%)
    - *Jersey ↔ Denim/Jeans* (US, UK, Canada)
    - → Strong evidence of **universal outfit-building logic**, with consistent and often high support across markets.

*   **Apparel ↔ Footwear (Consistent)**
    - *Dress → Shoes* (US, Canada)
    - *Shoes → Apparel* (Germany)
    - → Indicates **complete outfit purchasing behavior**, and their support signifies how common these patterns are in each region.

*   **Accessories Cross-Category (Less Consistent)**
    - *Bags → Shoes / Sunglasses* (US, Canada)
    - → Present but **not universal across markets**, suggesting that for other regions, these cross-category pairings may fall below the minimum support threshold.

**Cross-Country Similarities**

The analysis reveals **strong global consistency** in consumer behavior:

*   **Outfit Construction Logic**:
    - Consumers consistently buy **coherent outfits** (top + bottom + shoes)
    - These patterns often demonstrate good support across various countries**, affirming their commercial universality.

*   **Comfort/Home Bundles**:
    - Bathrobes and slippers are strongly linked across almost all countries
    - Consistently high support values make these universally actionable.

*   **Casualwear Clustering**:
    - T-shirts, sweatpants, sweatshirts frequently co-occur
    - Support for these clusters reinforces their general prevalence.

→ These patterns suggest **shared underlying consumption behavior across markets**, and their robust support values make them globally relevant.

**Key Cross-Country Differences**

Despite overall consistency, some **important differences emerge**:

*   **Strength of Associations**
    - The **US shows the richest and strongest patterns** (more rules, higher variety, and often higher support for a broader range of associations).
    - Smaller markets (e.g., Germany, Australia) show **fewer but still clear rules**, often with lower absolute support values due to smaller transaction volumes.

*   **Category Depth**
    - **Accessories are prominent in the US**, often with **sufficient support to generate rules**, but largely absent elsewhere where their co-occurrence might fall below thresholds.
    - Some niche categories (e.g., decorative items, stationery) appear only in specific countries, typically with **lower support** but high local relevance.

*   **Category vs Product Structure**
    - Some countries show stronger **category-level structure** (e.g., Canada, UK) with **higher support for category-level rules**.
    - Others rely more on **specific product-level relationships** where product-level rules have relatively higher support.

*   **Cultural / Usage Differences**
    - Variations in casualwear combinations (e.g., Germany’s *Informal Shirt–Short Pants*)
    - Differences likely reflect **climate, style preferences, or assortment**. The **support for these unique rules highlights their local significance.**

**Key Takeaways**

*   There is a **high degree of global consistency** in how customers build baskets:
    - Outfit-based logic dominates (top + bottom + footwear)
    - Comfort bundles are universal
    - **The recurring appearance and often robust support of these patterns underscore their global commercial viability.**

*   However, **local differences matter**:
    - Depth and richness of associations vary, often reflected in the **absolute support values across countries**.
    - Some categories (e.g., accessories) are market-specific, implying their **support is above threshold only in certain regions**.

*   **Managerial Implication**:
    - Use a **global recommendation backbone** (outfits, bundles) where **high support makes them universally applicable**.
    - Adapt locally for:
        - Category emphasis (e.g., accessories) where **local support allows**.
        - Strength of recommendations, modulated by **local support and lift**.
        - Market-specific patterns, identified by **rules with significant local support**.

Overall, consumer behavior is **structurally similar across countries**, but **differs in intensity, richness, and category focus**, with support serving as a key metric to quantify these cross-market variations and commercial potential.

## **8. Final Results**

The Market Basket Analysis highlights a set of strong and remarkably stable purchasing patterns across products, categories, time, and geographies. At the **product level**, the most robust association is the pairing between *Slippers and Bathrobes*, which consistently emerges as a high-support, high-confidence rule with very strong lift. This reflects a deeply embedded “home comfort” bundle that customers naturally perceive as a single consumption set. A second, more style-driven pattern appears with *Short Pants and Informal Shirts*, especially within the Versace Young segment, indicating a clear outfit-building logic already at the product level.

At a broader **category level**, this logic becomes even more evident. The *Pants–Shirt* combination dominates multi-item purchasing, confirming that top–bottom coordination is the backbone of basket construction. Similarly, the *SLG–Bags* relationship captures a meaningful accessory bundling behavior. In contrast, some high-frequency combinations—such as *Bags and Shoes*—do not reflect true affinity: despite appearing often together, their lift remains close to one, meaning the co-occurrence is driven by popularity rather than a genuine cross-selling opportunity.

Looking at the **temporal dimension**, the main bundles (homewear, outfits, accessories) remain stable even when expanding the observation window, suggesting that these are structural behaviors rather than transaction-specific coincidences. However, time-based analysis also reveals sequential dynamics in apparel purchases: customers often return to complete an outfit (e.g., buying a blouson first and then adding a t-shirt or shirt later), a pattern that would not be visible in single-basket analysis.

Segmentation further refines these insights. Versace Young stands out for its dense and high-support associations, making it particularly suitable for automated recommendation systems and “complete the look” strategies. In contrast, Versace Accessories shows strong complementarities only at the product level (e.g., within jewelry), with little evidence of category-level logic, implying that recommendations should remain highly specific rather than generic. Geographically, the core patterns—especially outfit construction and homewear bundles—are consistent across markets, although larger markets like the US exhibit richer and more diverse rule sets due to higher data volume.

**Geographically**, outfit-building logic (Top + Bottom + Footwear) and the Homewear bundle are globally consistent and should form the backbone of any universal recommendation engine. The US market shows the widest and richest rule set, including accessory cross-selling, while smaller markets (Germany, Australia) yield fewer rules due to smaller transaction volumes — not necessarily weaker behaviors

From a business perspective, these findings translate into a clear prioritization of actions. The *Bathrobe–Slippers* combination should be formalized into a “Home Comfort Set” and leveraged across digital merchandising, gifting, and CRM initiatives, given its exceptional reliability and scale. At the same time, “Complete the Look” recommendation engines should be actively deployed—especially for Versace Young—where strong and frequent outfit-based rules ensure both relevance and scalability.

For Versace Accessories, the strategy should shift toward SKU-level recommendations rather than category-level suggestions, reflecting how customers actually perceive and combine these products. Importantly, not all frequent combinations should be activated: pairs like *Bags and Shoes* should be excluded from recommendation logic, as their apparent association is not driven by real affinity and would dilute recommendation effectiveness.

Finally, temporal patterns open opportunities for CRM activation. Sequential purchase behaviors suggest that follow-up communications (e.g., after purchasing outerwear or footwear) can effectively prompt customers to complete their outfit. Overall, the strategy should rely on a global backbone of stable rules—outfit logic and homewear bundles—while allowing for local enrichment (e.g., stronger accessory cross-selling in high-volume markets).

## **Appendix - Style, Fabric & Color Analysis**

This session investigates co-purchase patterns at a highly granular level, focusing on **style, fabric, and color combinations** (extracted from the `style_fabric_color` column). The goal is to uncover very specific associations that might reveal insights into product design, merchandising, or customer preferences for coordinated items. Due to the high cardinality of these attributes, the analysis emphasizes the discovery of strong, often intra-product or intra-collection, complementarities.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STYLE, FABRIC & COLOR LEVEL ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

# 1. PRE-PROCESSING & CLEANING
# Standardize the detail column
df_complete['style_fabric_color'] = df_complete['style_fabric_color'].astype(str).str.strip().str.upper()

# Remove duplicate occurrences of the same style within a single transaction
df_style = df_complete.drop_duplicates(subset=["transaction_id", "style_fabric_color"]).copy()

# Keep only transactions with at least two distinct style/fabric/color combinations
style_counts = df_style.groupby("transaction_id")["style_fabric_color"].nunique()
multi_style_tids = style_counts[style_counts >= 2].index
df_style_multi = df_style[df_style["transaction_id"].isin(multi_style_tids)].copy()

print(f"Unique Style-Fabric-Colors: {df_complete['style_fabric_color'].nunique():,}")
print(f"Transactions with multi-style items: {len(multi_style_tids):,}")

# 2. BUILD BASKET MATRIX
basket_style = build_one_hot_basket(
    data=df_style_multi,
    transaction_col="transaction_id",
    item_col="style_fabric_color"
)

# 3. FINAL RULE GENERATION
# ADJUST THIS VALUE based on the "Support -> Rules" printout above.
# A good target for this level is roughly 50-100 high-quality rules.
MIN_SUPPORT_STYLE = 15 / len(multi_style_tids)
print(f"Calculated MIN_SUPPORT_STYLE: {MIN_SUPPORT_STYLE:.6f}")

rules_style = generate_rules(
    basket_matrix=basket_style,
    min_support=MIN_SUPPORT_STYLE,
    min_confidence=0.05,
    min_lift=1.0
)

# 4. FORMAT OUTPUT TABLES
summary_style = build_output_table(
    rules_df=rules_style,
    focal_name="focal_style",
    assoc_name="associated_style"
)

recommendations_style = build_recommendation_table(
    summary_df=summary_style,
    focal_col="focal_style",
    assoc_col="associated_style",
    top_n=3
)

print(f"\nFinal Style-level rules generated: {len(summary_style):,}")
display(recommendations_style.head(10))

This analysis explores co-purchase patterns at the **style level (style–fabric–color combinations)**, providing a highly granular view of customer behavior. **Due to the granularity, high lift values are often seen with very low support, which must be carefully interpreted.**

**Most Relevant Associations**

The results reveal **very strong and highly specific pairings**, mainly within the same product families. While lift is often exceptionally high, the **actual support (frequency) of these pairings can be very low**.

*   **Variant-Level Pairings (Same Product, Different Colors)**
    - *Athletic Socks (Black) ↔ Athletic Socks (White)* (e.g., `ATHLETIC SOCKS CALZA COTONE BIO MERCURY-BLACK+...` and `ATHLETIC SOCKS CALZA COTONE BIO MERCURY-WHITE+...`)
    - Probability ≈ 0.52–0.54, Support often very low (e.g., ~0.53%), **Lift ≈ 53**
    - → Customers frequently purchase **multiple color variants of the same item**, but the overall frequency (support) of these exact pairings is low at this granular level.

*   **Homewear Bundles**
    - *Bath Slippers ↔ Bathrobe* (same collection/style) (e.g., `BATH SLIPPERS I HEART BAROQUE-BLACK` and `BATH SLIPPERS I HEART BAROQUE-BLACK+BLACK`)
    - Example: *Baroque / Versace collections*
    - Probability ≈ 0.50–0.58, Support often very low (e.g., ~0.3% -0.7%), **Lift up to 100+**
    - → Strong complementarity within coordinated homewear sets. Again, high lift is seen, but actual market frequency (support) is typically minimal.

*   **Intra-Collection Consistency**
    - Strong links across items within the **same design line (e.g., “Baroque”, “Versace”)**
    - → Suggests customers buy **coordinated products within a collection**. These rules, while high in lift, typically have limited support, indicating niche or specialized purchasing.

However, some lift values are **extremely high (e.g., >50, >100)** and should be interpreted with caution. Potential explanations may be due to either:
-   **Systematically sold together** (e.g., bundled products, coordinated sets) where the **absolute support is still low**.
-   Or part of **very narrow assortments with low base frequency** for each item.
As a result high lift here may **not reflect true behavioral discovery**, but rather:
-   Predefined bundles
-   Merchandising strategies
-   Limited choice sets
It's crucial to acknowledge that the **commercial impact of these rules might be limited by their low support**.

**Key Takeaways**

*   The analysis captures **very granular and intuitive patterns**:
    - Multi-color purchasing of the same item
    - Strong within-collection consistency
    - Clear complementarity in homewear
    - These patterns, while insightful for product design, often have very low support and limited direct recommendation utility.

*   However, at this level:
    - Results are **less about cross-selling discovery of frequent patterns**
    - And more about **product design and merchandising structure**, as rules with high lift often coincide with very low support, signifying rare but strong connections.
    - **High lift with low support should be treated as a signal of strong inherent connection, but not necessarily broad market behavior.**